# 02 — Feature Engineering

**Hospital Readmission Risk Predictor**

This notebook demonstrates the feature engineering pipeline, focusing on ICD-9 code mapping,
medication change aggregation, and the clinical rationale behind each transformation.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_and_prepare
from src.leakage_detection import audit_leakage, remove_leakage_rows, generate_leakage_report
from src.feature_engineering import (
    map_icd9_to_category, add_diagnosis_categories,
    add_medication_flags, add_utilisation_score,
    encode_age_bracket, fill_race_unknown,
    drop_unwanted_columns, engineer_features
)
from src.utils import set_plot_style, PALETTE

set_plot_style()
print('Setup complete ✓')

In [ ]:
df = load_and_prepare()
print(f'Raw dataset: {df.shape}')

## 1. Leakage Detection & Removal

Before any feature engineering, we remove rows that represent information leakage. Patients who
expired or were discharged to hospice *cannot* be readmitted. Including them would give the model
a trivially predictable negative class.

In [ ]:
audit = audit_leakage(df)
for k, v in audit.items():
    print(f'  {k}: {v:,}')

In [ ]:
df = remove_leakage_rows(df)
report_path = generate_leakage_report(audit)
print(f'Leakage report saved to: {report_path}')
print(f'Dataset after leakage removal: {df.shape}')

## 2. ICD-9 Code Mapping

The raw `diag_1`, `diag_2`, `diag_3` columns contain ICD-9-CM codes — strings like `"428"` (heart
failure) or `"250.01"` (diabetes with ketoacidosis). These are too granular for a model to learn
from effectively, so we map them into ten clinically meaningful categories.

### Clinical Relevance

- **Circulatory** (390–459): Heart failure and stroke are leading drivers of readmission.
- **Diabetes** (250): The core condition for this cohort — primary diabetes diagnosis signals poor glycaemic control.
- **Respiratory** (460–519): COPD and pneumonia frequently cause early returns.
- **Mental Disorders** (290–319): Psychiatric comorbidities reduce medication adherence.

In [ ]:
# Demonstrate the mapping function
test_codes = ['250', '428', '486', '560', 'V58', 'E879', '296', '174', None]
for code in test_codes:
    category = map_icd9_to_category(code)
    print(f'  ICD-9 {str(code):>6s} → {category}')

In [ ]:
df = add_diagnosis_categories(df)

# Visualise primary diagnosis categories
diag1_counts = df['diag_1_cat'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
diag1_counts.plot(kind='barh', color=PALETTE[0], edgecolor='white', ax=ax)
ax.set_xlabel('Count')
ax.set_title('Primary Diagnosis Category Distribution')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Readmission rate by primary diagnosis category
diag_readmit = df.groupby('diag_1_cat')['readmit_30d'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
diag_readmit.plot(kind='barh', color=PALETTE[1], edgecolor='white', ax=ax)
ax.set_xlabel('30-Day Readmission Rate')
ax.set_title('Readmission Rate by Primary Diagnosis Category')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Medication Change Features

When a clinician adjusts medication during an encounter, it often signals clinical instability.
We aggregate all individual medication columns into two derived features:

- `n_med_changed` — how many diabetes medications had their dose changed
- `any_med_changed` — binary flag for any dosage adjustment

In [ ]:
df = add_medication_flags(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.groupby('readmit_30d')['n_med_changed'].mean().plot(
    kind='bar', ax=axes[0], color=[PALETTE[0], PALETTE[3]], edgecolor='white')
axes[0].set_title('Mean Medications Changed by Readmission')
axes[0].set_xticklabels(['No', 'Yes'], rotation=0)

df.groupby('any_med_changed')['readmit_30d'].mean().plot(
    kind='bar', ax=axes[1], color=[PALETTE[0], PALETTE[2]], edgecolor='white')
axes[1].set_title('Readmission Rate by Any Med Changed')
axes[1].set_xticklabels(['No Change', 'Changed'], rotation=0)

plt.tight_layout()
plt.show()

## 4. Prior Utilisation Score

Healthcare utilisation intensity is one of the strongest readmission predictors in the literature.
We combine prior inpatient, outpatient, and emergency visits into a single composite score.

In [ ]:
df = add_utilisation_score(df)

fig, ax = plt.subplots(figsize=(8, 5))
df.groupby('readmit_30d')['prior_visits_total'].mean().plot(
    kind='bar', color=[PALETTE[0], PALETTE[3]], edgecolor='white', ax=ax)
ax.set_title('Mean Prior Visits by Readmission Status')
ax.set_ylabel('Mean Total Prior Visits')
ax.set_xticklabels(['Not Readmitted', 'Readmitted <30d'], rotation=0)
plt.tight_layout()
plt.show()

## 5. Race Handling (Fairness-Aware)

For missing race values, we create an explicit `Unknown` category instead of imputing with the
statistical mode (which would be `Caucasian`). This decision preserves the fairness signal:
missingness in race data may correlate with socioeconomic factors that themselves affect readmission risk.

In [ ]:
n_missing_before = df['race'].isna().sum()
df = fill_race_unknown(df)
n_unknown = (df['race'] == 'Unknown').sum()
print(f'Missing race values filled → "Unknown": {n_missing_before:,} → {n_unknown:,}')
print(f'\nRace distribution after fill:')
display(df['race'].value_counts())

## 6. Full Pipeline Output

Let's run the complete feature engineering pipeline and inspect the final feature set.

In [ ]:
# Reload and run full pipeline
df_raw = load_and_prepare()
df_raw = remove_leakage_rows(df_raw)
df_final = engineer_features(df_raw)

print(f'Final dataset shape: {df_final.shape}')
print(f'\nColumns ({len(df_final.columns)}):')
for col in df_final.columns:
    dtype = df_final[col].dtype
    n_unique = df_final[col].nunique()
    print(f'  {col:30s}  dtype={str(dtype):10s}  unique={n_unique}')

## Summary

Feature engineering produces a clean, clinically-grounded feature set:

| Transformation | Features Created | Rationale |
|---|---|---|
| ICD-9 mapping | `diag_1_cat`, `diag_2_cat`, `diag_3_cat` | Clinically meaningful categories instead of thousands of raw codes |
| Medication flags | `n_med_changed`, `any_med_changed` | Captures clinical instability signal |
| Utilisation score | `prior_visits_total` | Strongest predictor per literature |
| Age encoding | `age_ordinal` | Preserves ordinal relationship |
| Race fill | `Unknown` category | Fairness-preserving approach |

The raw ICD-9 codes and individual medication columns are dropped after aggregation to keep
dimensionality manageable.